# 📘 skimage 세분화와 특징 추출

이미지 세분화(segmentation)는 이미지를 의미 있는 영역으로 나누는 작업입니다.
특징 추출(feature extraction)은 객체의 형태, 면적, 둘레 등을 계산합니다.

**학습 목표:**
- 라벨링과 영역 특징 추출
- 세분화 기법 (임계값, SLIC, 활성등고선)
- 객체 탐지와 속성 측정
- 히스토그램 명암 보정

## 1. 라벨링과 영역 특징 추출

이진화된 이미지에서 객체를 식별하고, 각 객체의 면적, 둘레, 중심 등을 계산합니다.
`measure.label()`로 객체를 라벨링하고, `measure.regionprops()`로 속성을 추출합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  라벨링과 영역 특징 추출                  │
# │  measure.label, regionprops              │
# └─────────────────────────────────────────┘

from skimage import data, measure, color, morphology
from skimage.filters import threshold_otsu
import matplotlib.pyplot as plt
import numpy as np

# 동전 이미지
img = data.coins()

# 이진화
thresh = threshold_otsu(img)
binary = img > thresh
binary_cleaned = morphology.remove_small_objects(binary, min_size=100)
binary_cleaned = morphology.remove_small_holes(binary_cleaned, area_threshold=100)

# 라벨링
labels = measure.label(binary_cleaned)
num_objects = labels.max()
print(f'검출된 객체 수: {num_objects}')

# 영역 특징 추출
props = measure.regionprops(labels)
print(f'\n=== 동전 분석 ===')
print(f'{"번호":>4s} {"면적":>8s} {"둘레":>8s} {"중심":>12s} {"반경":>8s}')
print('-' * 48)
for i, prop in enumerate(props):
    print(f'{i+1:>4d} {prop.area:>8.0f} {prop.perimeter:>8.1f} '
          f'({prop.centroid[0]:>5.1f},{prop.centroid[1]:>5.1f}) {prop.equivalent_diameter_area:>8.1f}')

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('원본')
axes[1].imshow(binary_cleaned, cmap='gray'); axes[1].set_title('이진화 (노이즈 제거)')
axes[2].imshow(color.label2rgb(labels, image=img, bg_label=0))
axes[2].set_title(f'라벨링 ({num_objects}개 객체)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print('\n💡 measure.label(): 연결된 객체에 고유 번호 부여')
print('💡 regionprops(): 면적, 둘레, 중심, 바운딩박스 등 속성 추출')
print('💡 remove_small_objects/hodes: 작은 노이즈/구멍 제거')

## 2. SLIC 초화소 세분화

**SLIC**(Simple Linear Iterative Clustering)은 이미지를 색상과 공간 정보를 기반으로
유사한 영역(초화소, superpixel)으로 나눕니다. 과세분화(over-segmentation)에 유용합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  SLIC 초화소 세분화                      │
# │  색상+공간 정보 기반 클러스터링           │
# └─────────────────────────────────────────┘

from skimage import data, segmentation, color
from skimage.segmentation import mark_boundaries

img = data.astronaut()

# SLIC 세분화 (segments 수 지정)
segments_50 = segmentation.slic(img, n_segments=50, compactness=10)
segments_100 = segmentation.slic(img, n_segments=100, compactness=10)
segments_200 = segmentation.slic(img, n_segments=200, compactness=10)

print(f'segments=50:  {segments_50.max()+1}개 초화소')
print(f'segments=100: {segments_100.max()+1}개 초화소')
print(f'segments=200: {segments_200.max()+1}개 초화소')

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(img); axes[0].set_title('원본')
axes[1].imshow(mark_boundaries(img, segments_50))
axes[1].set_title(f'SLIC (n=50, {segments_50.max()+1}개)')
axes[2].imshow(mark_boundaries(img, segments_100))
axes[2].set_title(f'SLIC (n=100, {segments_100.max()+1}개)')
axes[3].imshow(mark_boundaries(img, segments_200))
axes[3].set_title(f'SLIC (n=200, {segments_200.max()+1}개)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print('💡 n_segments: 원하는 세분화 영역 수 (실제 결과는 약간 다를 수 있음)')
print('💡 compactness: 색상 vs 공간의 균형 (클수록 경계가 더 규칙적)')
print('💡 mark_boundaries: 세분화 경계를 원본 이미지 위에 표시')

## 3. 히스토그램 명암 보정

이미지의 대비를 개선하는 기법입니다.
`exposure` 모듈은 히스토그램 평활화, 대비 스트레칭 등을 제공합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  히스토그램 명암 보정                    │
# │  equalize_hist, equalize_adapthist       │
# └─────────────────────────────────────────┘

from skimage import data, exposure, util
import numpy as np

# 저대비 이미지 생성
img = data.camera()
img_low = exposure.rescale_intensity(img, in_range=(80, 180))  # 저대조

# 히스토그램 평활화
img_eq = exposure.equalize_hist(img_low)
img_eq = util.img_as_ubyte(img_eq)

# 적응형 히스토그램 평활화 (CLAHE)
img_adapteq = exposure.equalize_adapthist(img_low, clip_limit=0.03)
img_adapteq = util.img_as_ubyte(img_adapteq)

# 감마 보정
img_gamma = exposure.adjust_gamma(img_low, gamma=0.5)  # 밝게

# 대비 스트레칭
img_stretch = exposure.rescale_intensity(img_low)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
images = [img, img_low, img_eq, img_adapteq, img_gamma]
titles = ['원본', '저대비', '평활화', 'CLAHE', '감마(0.5)']
for col, (im, t) in enumerate(zip(images, titles)):
    axes[0, col].imshow(im, cmap='gray')
    axes[0, col].set_title(t)
    axes[0, col].axis('off')
    # 히스토그램
    axes[1, col].hist(im.ravel(), bins=256, range=(0, 256), color='steelblue')
    axes[1, col].set_title(f'{t} 히스토그램')
plt.tight_layout()
plt.show()

print('💡 equalize_hist(): 전체 히스토그램 평활화 (전역)')
print('💡 equalize_adapthist(): 적응형 평활화 (국소, CLAHE)')
print('💡 adjust_gamma(): 감마 보정 (gamma<1: 밝게, gamma>1: 어둡게)')
print('💡 rescale_intensity(): 대비 스트레칭 (최소/최대를 0~255로)')

## 4. 객체 탐지와 속성 시각화

검출된 객체의 바운딩 박스, 중심점, 방향 등을 시각화합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  객체 탐지와 속성 시각화                 │
# │  regionprops 속성을 이미지 위에 표시      │
# └─────────────────────────────────────────┘

img = data.coins()
thresh = threshold_otsu(img)
binary = img > thresh
binary = morphology.remove_small_objects(binary, min_size=50)
binary = morphology.remove_small_holes(binary, area_threshold=50)

labels = measure.label(binary)
props = measure.regionprops(labels)

# 결과 이미지 (컬러)
result = color.gray2rgb(img)

for prop in props:
    # 바운딩 박스
    minr, minc, maxr, maxc = prop.bbox
    rr, cc = np.array([minr, minr, maxr, maxr, minr]), \
             np.array([minc, maxc, maxc, minc, minc])
    result[rr, cc] = [255, 0, 0]  # 빨간 박스

    # 중심점
    cr, cc_center = prop.centroid
    r, c = int(cr), int(cc_center)
    result[max(0,r-3):r+4, max(0,c-3):c+4] = [0, 255, 0]  # 초록 점

print(f'검출된 객체 수: {len(props)}')
print(f'\n=== 동전 속성 ===')
print(f'{"번호":>4s} {"면적":>8s} {"등가지름":>8s} {"방향":>8s} {"종횡비":>8s}')
print('-' * 44)
for i, prop in enumerate(props):
    print(f'{i+1:>4d} {prop.area:>8.0f} {prop.equivalent_diameter_area:>8.1f} '
          f'{np.degrees(prop.orientation):>8.1f} {prop.major_axis_length/prop.minor_axis_length:>8.2f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('원본')
axes[1].imshow(result); axes[1].set_title(f'객체 탐지 ({len(props)}개)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print('\n💡 bbox: 바운딩 박스 (min_row, min_col, max_row, max_col)')
print('💡 centroid: 객체의 무게 중심 (행, 열)')
print('💡 orientation: 객체의 주축 방향 (라디안)')
print('💡 equivalent_diameter_area: 면적이 같은 원의 지름')

## 🎯 연습 문제

1. `data.coins()` 이미지에서 면적이 가장 큰 동전 3개만 강조 표시하세요.
2. SLIC 세분화에서 `compactness` 매개변수를 5, 10, 20으로 변경하며 결과를 비교하세요.
3. 저대비 이미지에 대해 전역 평활화와 CLAHE를 적용하고, 히스토그램 변화를 비교하세요.
4. `regionprops()`에서 `eccentricity`(이심률) 속성을 사용해 원형 객체와 타원형 객체를 구분하세요.
5. `data.page()` 이미지에 적응형 임계값 → 라벨링 → regionprops 파이프라인을 적용하세요.